In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install xgboost
import pandas as pd
import numpy as np
import seaborn as sb
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb # Now xgboost should be available
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor

In [3]:
import gdown

file_id = "1mYa7dqccu2Ha7g5dPvuJGwSWFN48GP1T"

url = f"https://drive.google.com/uc?id={file_id}"
output = "test_x.csv"
gdown.download(url, output, quiet=False)

# Now read the downloaded CSV file
X_test = pd.read_csv(output, header=0, sep=',')

Downloading...
From: https://drive.google.com/uc?id=1mYa7dqccu2Ha7g5dPvuJGwSWFN48GP1T
To: /content/test_x.csv
100%|██████████| 2.91M/2.91M [00:00<00:00, 88.5MB/s]


In [4]:
data_path = "https://drive.google.com/uc?id=1W9Rs9UTFSOYGDww_12slH7s3m3UpyvM4&export=download"
X_train = pd.read_csv(data_path, header=0, sep=',')
data_path = "https://drive.google.com/uc?id=1rrAgZbWumue9kUDiQDPD5_Mw48RSgmhD&export=download"
Y_train = pd.read_csv(data_path)

In [5]:
X_train['index']= X_train['date'] +'_'+ X_train['station']
index_test=X_test['index']
data_train = X_train.merge(Y_train,on = 'index')
d_train = data_train

# numeroté les stations
label_encoder = LabelEncoder()
d_train['station'] = label_encoder.fit_transform(d_train['station'])
X_test['station'] = label_encoder.transform(X_test['station'])

#X_test['station2'] = label_encoder.transform(X_test['station'])
df_train= d_train.drop(columns=['index']) # sans l'index
df_test= X_test.drop(columns=['index']) # sans l'index

# Fonction pour exporter les données sur le site datachallenge
def export(predictions, index_test, filename='predictions.csv'):

    predictions_df = pd.DataFrame({'index': index_test, 'y': predictions})


    predictions_df.to_csv(filename, index=False, sep=',')
    print(f"Predictions exported to {filename} with delimiter ','")

df_train['y_log'] = np.log1p(df_train['y'])  # log(1 + y) pour éviter log(0)
df_train

,date,station,job,ferie,vacances,y,y_log
0,2015-01-01,24,1,1,1,7,2.079442
1,2015-01-01,295,1,1,1,0,0.000000
2,2015-01-01,107,1,1,1,9,2.302585
3,2015-01-01,367,1,1,1,9,2.302585
4,2015-01-01,199,1,1,1,28,3.367296
...,...,...,...,...,...,...,...
1229858,2022-12-31,368,0,0,1,1227,7.113142
1229859,2022-12-31,285,0,0,1,544,6.300786
1229860,2022-12-31,310,0,0,1,92,4.532599
1229861,2022-12-31,143,0,0,1,91,4.521789


In [6]:
df_test=df_test.drop(columns='date')

X = df_train.drop(columns=['y','y_log','date'])
y = df_train['y_log']
X

,station,job,ferie,vacances
0,24,1,1,1
1,295,1,1,1
2,107,1,1,1
3,367,1,1,1
4,199,1,1,1
...,...,...,...,...
1229858,368,0,0,1
1229859,285,0,0,1
1229860,310,0,0,1
1229861,143,0,0,1


In [ ]:
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
# Initialisation des nouveaux modèles
models = {
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=42),#
    "ExtraTreeRegressor": ExtraTreeRegressor(random_state=42),
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=42),#
    "RandomForestRegressor": RandomForestRegressor(random_state=42),#
    "BaggingRegressor": BaggingRegressor(random_state=42)
}

# Entraînement et prédiction pour chaque modèle
predictions = {}
for name, model in models.items():
    model.fit(X, y)
    y_pred = model.predict(df_test)
    y_predd=np.expm1(y_pred)
    predictions[name] = y_predd
    print(f"{name}: Prédiction sur X_test = {y_predd[:5]}")

ExtraTreesRegressor: Prédiction sur X_test = [19.84076244 11.77934489 13.47935058  1.60255412 30.18727172]
ExtraTreeRegressor: Prédiction sur X_test = [19.84076244 11.77934489 13.47935058  1.60255412 30.18727172]
DecisionTreeRegressor: Prédiction sur X_test = [19.84076244 11.77934489 13.47935058  1.60255412 30.18727172]
RandomForestRegressor: Prédiction sur X_test = [20.4365187  11.68397024 13.57147742  1.5455484  30.38629251]
BaggingRegressor: Prédiction sur X_test = [19.52776008 12.09690271 13.78150276  1.34058609 30.92303942]


In [8]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
df_train['dayofweek'] = pd.to_datetime(df_train['date']).dt.dayofweek
df_train['month'] = pd.to_datetime(df_train['date']).dt.month
numerical_features = ['dayofweek', 'month', 'station', 'job', 'ferie']
categorical_features = []

numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', ExtraTreesRegressor(random_state=42))
])
param_grid = {
    'regressor__n_estimators': [100],
    'regressor__max_depth': [None, 10],
}

X = df_train[['dayofweek', 'month', 'station', 'job', 'ferie']]
y = df_train['y_log']
grid_search = GridSearchCV(pipeline, param_grid, scoring='neg_mean_squared_error', cv=5, n_jobs=-1)
grid_search.fit(X, y)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(df_test)
y_predd = np.expm1(y_pred)
predictions[name] = y_predd

print(f"{name}: Meilleurs paramètres = {grid_search.best_params_}")
print(f"{name}: Prédiction sur X_test = {y_predd[:5]}")

KeyboardInterrupt: 

In [9]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from joblib import Parallel, delayed, parallel_backend # Importez parallel_backend


# Création des features
df_train['dayofweek'] = pd.to_datetime(df_train['date']).dt.dayofweek
df_train['month'] = pd.to_datetime(df_train['date']).dt.month

# Définition des features numériques et catégorielles
numerical_features = ['dayofweek', 'month', 'station', 'job', 'ferie']
categorical_features = []

# Pipeline de prétraitement
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Pipeline complet avec le modèle
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', ExtraTreesRegressor(random_state=42))
])

# Grille de paramètres pour GridSearchCV
param_grid = {
    'regressor__n_estimators': [100, 200, 300], # Exemple de paramètres
    'regressor__max_depth': [None, 10, 20, 30],
    # ... autres paramètres ...
}

# Changement principal : utilisation de 'loky' et un nombre de jobs plus conservateur
with parallel_backend('loky', n_jobs=4): # ou n_jobs=-2 pour utiliser tous les cœurs sauf un
    grid_search = GridSearchCV(pipeline, param_grid, scoring='neg_mean_squared_error', cv=5)
    grid_search.fit(df_train[numerical_features], df_train['y_log'])

# ... (le reste de votre code) ...

KeyboardInterrupt: 

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(df_test[numerical_features]) # Prédictions sur les données de test
y_predd = np.expm1(y_pred) # Transformation inverse si vous avez utilisé np.log1p

In [ ]:
export(y_predd, index_test, filename='predictions.csv') # Utilisation de votre fonction 'export'

In [ ]:
print(f"Meilleurs paramètres : {grid_search.best_params_}")
print(f"Score du meilleur modèle : {grid_search.best_score_}")


In [ ]:

best_model = grid_search.best_estimator_
y_pred = best_model.predict(df_test[numerical_features])
y_predd = np.expm1(y_pred)

export(y_predd, index_test, filename='predictions.csv')

print(f"Meilleurs paramètres : {grid_search.best_params_}")
print(f"Score du meilleur modèle : {grid_search.best_score_}")

In [ ]:
# pour creer le fichier à soumettre
pred_dec=predictions["DecisionTreeRegressor"]
export(pred_dec, index_test,'y_pred_dec.csv')

Predictions exported to y_pred_dec.csv with delimiter ','


In [ ]:
from google.colab import files

files.download('y_pred_dec.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.ensemble import BaggingRegressor, ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
import numpy as np

# Définir les modèles et leurs grilles d'hyperparamètres
models = {
    "ExtraTreesRegressor": {
        "model": ExtraTreesRegressor(random_state=42),
        "params": {
            'n_estimators': [100, 200, 300],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
        }
    },
    "ExtraTreeRegressor": {
        "model": ExtraTreeRegressor(random_state=42),
        "params": {
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
        }
    },
    "DecisionTreeRegressor": {
        "model": DecisionTreeRegressor(random_state=42),
        "params": {
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
        }
    },
    "RandomForestRegressor": {
        "model": RandomForestRegressor(random_state=42),
        "params": {
            'n_estimators': [100, 200, 300],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
        }
    },
    "BaggingRegressor": {
        "model": BaggingRegressor(random_state=42),
        "params": {
            'n_estimators': [10, 20, 30],
            'max_samples': [0.5, 0.7, 1.0],
        }
    }
}

# Entraînement et prédiction avec GridSearchCV
predictions = {}
for name, model_data in models.items():
    model = model_data["model"]
    params = model_data["params"]

    grid_search = GridSearchCV(model, params, scoring='neg_mean_squared_error', cv=5, n_jobs=-1)
    grid_search.fit(X, y)

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(df_test)
    y_predd = np.expm1(y_pred)
    predictions[name] = y_predd

    print(f"{name}: Meilleurs paramètres = {grid_search.best_params_}")
    print(f"{name}: Prédiction sur X_test = {y_predd[:5]}")